In [2]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Forzar el cierre de cualquier sesión previa para liberar el puerto
try:
    spark.stop()
except Exception:
    pass

# Iniciar SparkSession usando el puerto 4042 (el único abierto en Docker)
spark = SparkSession.builder \
    .appName("Benchmark-Kappa-Streaming") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

print("=== INICIANDO BENCHMARK: KAPPA (STREAMING EN VIVO) ===")

# Simular origen de datos en Streaming (10k registros por segundo)
df_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10000) \
    .load()

# Transformación al vuelo
df_transformed = df_stream \
    .withColumn("sensor_id", (col("value") % 100).cast("string")) \
    .withColumn("temperatura", (col("value") % 50) + 15.5)

# Salida por consola para ver los micro-lotes fluyendo
query = df_transformed.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

print("🌊 Flujo Kappa iniciado. Procesando en tiempo real...")
print("🔍 Manteniendo el streaming activo por 20 segundos...")

# Dejar correr el flujo por 20 segundos para capturar la evidencia
time.sleep(20)

# Detener el stream ordenadamente
query.stop()
print("✅ Stream finalizado correctamente.")

=== INICIANDO BENCHMARK: KAPPA (STREAMING EN VIVO) ===
🌊 Flujo Kappa iniciado. Procesando en tiempo real...
🔍 Manteniendo el streaming activo por 20 segundos...


26/10/19 03:31:43 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0a41d223-4a85-4d21-bafa-49eb29d9072e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/10/19 03:31:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+---------+-----------+
|timestamp|value|sensor_id|temperatura|
+---------+-----+---------+-----------+
+---------+-----+---------+-----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-----+---------+-----------+
|           timestamp|value|sensor_id|temperatura|
+--------------------+-----+---------+-----------+
|2026-10-19 03:31:...|    0|        0|       15.5|
|2026-10-19 03:31:...|    8|        8|       23.5|
|2026-10-19 03:31:...|   16|       16|       31.5|
|2026-10-19 03:31:...|   24|       24|       39.5|
|2026-10-19 03:31:...|   32|       32|       47.5|
|2026-10-19 03:31:...|   40|       40|       55.5|
|2026-10-19 03:31:...|   48|       48|       63.5|
|2026-10-19 03:31:...|   56|       56|       21.5|
|2026-10-19 03:31:...|   64|       64|       29.5|
|2026-10-19 03:31:...|   72|     

26/10/19 03:32:03 WARN DAGScheduler: Failed to cancel job group 45938dbc-a66b-4636-8d63-421602e46ed3. Cannot find active jobs for it.
26/10/19 03:32:03 WARN DAGScheduler: Failed to cancel job group 45938dbc-a66b-4636-8d63-421602e46ed3. Cannot find active jobs for it.
